# Dataset Exploration and Validation

## Research Project: Probing Syntactic Structure in Transformers

### Objective
This notebook explores the Universal Dependencies English Web Treebank (UD English-EWT) and checks whether its syntactic annotations are structurally consistent before using them in your Transformer probing project.

Planned steps
1. Load and inspect the dataset.
2. Understand token indices and dependency heads.
3. Validate annotation lengths.
4. Check head indices, roots, and dependency cycles.
5. Summarize the results.


In [ ]:
#Loading a small sample 
from datasets import load_dataset
import pandas as pd

#Load first five sentences from UD English-EWT
sample_dataset = load_dataset(
    "universal-dependencies/universal_dependencies",
    "en_ewt",
    split="train[:5]",
)

print(sample_dataset)
print("\n First example:")
print(sample_dataset[0])

## Understand token indices
Universal Dependencies uses token indices starting at 1, while pythn lists start at 0. The dependency head annotations use UD indices, so we must handle the difference carefully.

In [ ]:
# Inspect the first sentence.
example = sample_dataset[0]

print("Sentence:", example["text"])
print()

print(f'{"UD index":<10} {"Python index":<13} {"Token"}')
print("-" * 40)

for ud_index, token in enumerate(example["tokens"], start=1):
    python_index = ud_index - 1
    print(f"{ud_index:<10} {python_index:<13} {token}")

In [ ]:
#checking the head of a specific word

word = "American"

tokens= example["tokens"]
heads = example["head"]
relations = example["deprel"]

#find the words positions
word_index = tokens.index(word)

#convert Python index to UD index
ud_index = word_index + 1

#get its head index
head_index = heads[word_index]

print("Word:", word)
print("Pytho index:", word_index)
print("UD index:", ud_index)
print("Head index:", head_index)

#Find the head word
if head_index == 0:
    head_word = "ROOT"
else:
    head_word = tokens[head_index - 1]

print("Head word:", head_word)
print("Dependency relation:", relations[word_index])

In [ ]:
#inspecting every word and its head
print(f" {'UD idx':<8} {'Token':<12} {'Head idx':<10} {'Head word':<12} {'Relation'}")
print("-" * 65
)
for i, (token, head, relation) in enumerate(
    zip(example["tokens"], example["head"], example["deprel"]),
    start = 1,
):

    head_word = "ROOT" if head == 0 else example["tokens"][head - 1]

    print(f"{i:<8} {token:<12} {head:<10} {head_word:<12} {relation}"
)

In [ ]:
# Print the annotation lists

print("Tokens:", example["tokens"])
print("POS tags:", example["upos"])
print("Head indices:", example["head"])
print("Dependencies:", example["deprel"])

## Inspect dependency annotations

This section shows each token, its part-of-speech tag, its head, and its dependency relation.

In [ ]:
example = sample_dataset[0]
tokens = example["tokens"]

# Convert each head index to its corresponding word.
head_words = []

for head_index in example["head"]:
    if head_index == 0:
        head_words.append("ROOT")
    elif 1 <= head_index <= len(tokens):
        head_words.append(tokens[head_index - 1])
    else:
        head_words.append("[head index out of range]")

annotation_table = pd.DataFrame({
    "UD index": range(1, len(tokens) + 1),
    "Python index": range(len(tokens)),
    "Token": tokens,
    "UPOS": example["upos"],
    "Head index": example["head"],
    "Head word": head_words,
    "Dependency relation": example["deprel"],
})

display(annotation_table)

In [ ]:
#Displaying all 5 sentences using pandas

for sentence_num, example in enumerate(sample_dataset, start=1):
    print(f"\nSENTENCE {sentence_num}: {example['text']}")

    table = pd.DataFrame({
        "UD index": range(1, len(example["tokens"]) + 1),
        "Token": example["tokens"],
        "POS": example["upos"],
        "Head index": example["head"],
        "Dependency relation": example["deprel"],
    })

    table["Head word"] = [
        "ROOT" if head == 0 else example["tokens"][head - 1]
        for head in example["head"]
    ]

    display(table)

In [ ]:
print("Number of sentences:", len(sample_dataset))
print("Number of tokens in sentence 1:", len(sample_dataset[0]["tokens"]))

In [ ]:
# check whetehr dependency annotaions align
example = sample_dataset[0]

print("Tokens:", len(example["tokens"]))
print("POS tags:", len(example["upos"]))
print("Head indices:", len(example["head"]))
print("Dependency relations:", len(example["deprel"]))

In [ ]:
#Checking for every sentence, not just the first 

for sentence_num, example in enumerate(sample_dataset, start=1):
    print(
        sentence_num,
        len(example["tokens"]),
        len(example["upos"]),
        len(example["head"]),
        len(example["deprel"]),
    )

## Loading full training dataset


In [ ]:


train_dataset = load_dataset(
    "universal-dependencies/universal_dependencies",
    "en_ewt",
    split="train",
)

print("Training sentences:", len(train_dataset))

In [ ]:
#checking full training dataset

print("Number of training sentences:", len(train_dataset))

# Check the first sentence
example = train_dataset[0]

print("\nFirst sentence:")
print(example["text"])

print("\nNumber of tokens:", len(example["tokens"]))
print("Number of POS tags:", len(example["upos"]))
print("Number of head indices:", len(example["head"]))
print("Number of dependency relations:", len(example["deprel"]))

In [ ]:
# Check annotation consistency across the full dataset

mismatches = []

for sentence_num, example in enumerate(train_dataset, start=1):
    token_count = len(example["tokens"])

    if not (
        token_count == len(example["upos"])
        == len(example["head"])
        == len(example["deprel"])
    ):
        mismatches.append(sentence_num)

print("Total sentences checked:", len(train_dataset))
print("Sentences with mismatched annotation lengths:", len(mismatches))
print("First mismatched sentence numbers:", mismatches[:10])

In [ ]:
# Validate the head indices

invalid_heads = []

for sentence_num, example in enumerate(train_dataset, start=1):
    token_count = len(example["tokens"])

    for token_index, head in enumerate(example["head"], start=1):
        if head < 0 or head > token_count:
            invalid_heads.append({
                "sentence": sentence_num,
                "token_index": token_index,
                "head": head,
                "token": example["tokens"][token_index - 1],
            })

print("Invalid head indices:", len(invalid_heads))
print("First invalid examples:", invalid_heads[:5])

In [ ]:
# check number of roots per sentence

root_issues = []

for sentence_num, example in enumerate(train_dataset, start=1):
    root_count = example["head"].count(0)

    if root_count != 1:
        root_issues.append({
            "sentence": sentence_num,
            "root_count": root_count,
            "text": example["text"],
        })

print("Sentences checked:", len(train_dataset))
print("Sentences without exactly one root:", len(root_issues))
print("First root issues:", root_issues[:5])

In [ ]:
# Dependency Cycle check

def has_cycle(heads):
    for start_token in range(1, len(heads) + 1):
        visited = set()
        current_token = start_token

        while current_token != 0:
            if current_token in visited:
                return True

            visited.add(current_token)

            next_token = heads[current_token - 1]
            current_token = next_token

    return False

print("Cyclic example:", has_cycle([2, 1, 0]))
print("Valid example:", has_cycle([0, 1, 2]))

In [ ]:
# check full training split
sentences_with_cycles = []

for sentence_num, example in enumerate(train_dataset, start=1):
    if has_cycle(example["head"]):
        sentences_with_cycles.append(sentence_num)

print("Total sentences checked:", len(train_dataset))
print("Sentences with cycles:", len(sentences_with_cycles))
print("First sentences with cycles:", sentences_with_cycles[:10])

## Calculate sentence lengths
# Sentence Length Analysis
We examine the number of tokens in each sentence to understand the length distribution of the UD English-EWT training dataset. This helps us understand the data and plan batches for extracting BERT representations.


In [ ]:
#Calculate the number of tokens in each training sentence
sentence_lengths = [
    len(example["tokens"])
    for example in train_dataset
]

#Display basic statistics about sentence lengths
print("Number of sentences:", len(sentence_lengths))
print("Shortest sentence:", min(sentence_lengths), "tokens")
print("Longest sentence:", max(sentence_lengths), "tokens")
print("Average sentence length:", round(sum(sentence_lengths) / len(sentence_lengths), 2), "tokens")

In [ ]:
import matplotlib.pyplot as plt

plt.hist(sentence_lengths, bins= 30)
plt.xlabel("Sentence length (tokens)")
plt.ylabel("Number of sentences")
plt.title("Sentence length Distribution in UD English-EWT")
plt.show()

In [ ]:
import statistics
median_length = statistics.median(sentence_lengths)
print("Median sentence length:", median_length, "tokens")

In [ ]:
#Verifying observartion numerically

short_sentences = sum(
    length <= 10 for length in sentence_lengths
)
percentage = short_sentences / len(sentence_lengths) * 100
print("Sentnces with 10 tokens or fewer:", short_sentences)
print("Percentage:", round(percentage, 2), "%")

# Sentence Length Distribution
The UD English-EWT training split contains 12,544 sentences. Sentence lengths range from 1 to 159 tokens, with a mean of 16.31 tokens and a median of 14 tokens. The histogram suggests a right-skewed distribution, with many shorter sentences and a tail extending toward longer sentences. A total of 4,661 sentences(37.16%) contain 10 tokens or fewer. This analysis provides an initial understanding of sentence lengths before extracting BERT representations.

In [ ]:
## Check POS label distribution
from collections import Counter
pos_counts = Counter(
    tag
    for example in train_dataset
    for tag in example["upos"]
)

print(pos_counts.most_common(10))
print("Number of unique POS tags:", len(pos_counts))
print("Least frequent POS tags:")
print(sorted(pos_counts.items(), key=lambda item: item[1])[:5])

POS Tag Distribution
The training split contains 17 unique Universal POS tags. NOUN is the most frequent tag, with 34,751 occurences, while X is the least frequent among the observed tags, with 400 occurences. Other relatively infrequent tags include INTJ(695) and SYM(746).

The diftribution is imbaalnced, with substantial differences in the frequencies of grammatical categories. This should be considered when evaluating a POS probing classifier. In addition to accuracy, macro-F1 and per-class performance will help assess how well the probe predicts both frequent and infrequent tags.


# Check the dependency relation labels
POS tags describe grammatical categories, such as nouns and verbs. Dependency 
relations describe how words relate to one another in a sentence - for example,
a subject, object, or modifier. 

In [ ]:
deprel_counts = Counter(
    relation
    for example in train_dataset
    for relation in example["deprel"]
)
print("Number of unique dependency relations:", len(deprel_counts))
print("Most frequent relations:", deprel_counts.most_common(10))
print("Least frequent dependency relations:")
print(sorted(deprel_counts.items(), key=lambda item: item[1])[:5])

## Dependency Relation Distribution
The UD English-EWT training split contains 51 unique dependency relation labels. The most frequent relation is punct (23,556 occurences), while csubj:outer is the least frequent among the observed labels(2 occurences). Other rare relations include dep(3), dislocated(5), and csubj:pass(7).
This substantial variation in label frequencies indicates class imbalance, which may affect dependency relation classification. Evaluation should therefore include macro-F1 and per-class results in addition to overall accuracy. The rarity of some labels will also need to be considered when designing the probing experiment. 

## Load the validation dataset

In [ ]:
validation_dataset = load_dataset(
    "universal-dependencies/universal_dependencies",
    "en_ewt",
    split="dev",
)
print("Validation sentences:", len(validation_dataset))

## Load the test dataset

In [ ]:

test_dataset = load_dataset(
    "universal-dependencies/universal_dependencies",
    "en_ewt",
    split="test",
)
print("Test sentences:", len(test_dataset))

In [ ]:
# Verify all three splits 
print("Train:", len(train_dataset))
print("Validation:", len(validation_dataset))
print("Test:", len(test_dataset))
print("Total:", len(train_dataset) + len(validation_dataset) + len(test_dataset))

In [ ]:
#importing libraries and load the tokenizer
import torch
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


In [ ]:
# Loading the Pretrained BERT and select the device

from transformers import AutoModel

model = AutoModel.from_pretrained("bert-base-uncased")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Selected device", device)
print("Hidden size:", model.config.hidden_size)
print("Number of BERT layers:", model.config.num_hidden_layers)
print("Model device:", next(model.parameters()).device)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
#Checking BERT tokenization

example = train_dataset[0]
words = example["tokens"]

encoded = tokenizer(
    words,
    is_split_into_words=True,
    add_special_tokens=True,
)

print("Original words:")
print(words)

print("\nBERT tokens:")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"]))

In [ ]:
#Finding original word positions

word_ids = encoded.word_ids()

print("Original words:")
print(words)

print("\nBERT tokens and their original word indices:")
for token, word_id in zip(
    tokenizer.convert_ids_to_tokens(encoded["input_ids"]),
    word_ids
):
    print(f"{token:12} -> {word_id}")

In [ ]:
# Identify only the first subword of each word

first_subword_positions = []
previous_word_id = None

for position, word_id in enumerate(word_ids):
    if word_id is not None and word_id != previous_word_id:
        first_subword_positions.append(position)

    previous_word_id = word_id

print("Number of original words:", len(words))
print("Number of first subwords:", len(first_subword_positions))
print("First-subword positions:", first_subword_positions)

In [ ]:
#Align the POS labels

pos_tags = example["upos"]

print("Number of original words:", len(words))
print("Number of POS labels:", len(pos_tags))

print("\nWord "   " -> "   "  POS tag "   " -> "   " BERT first subword")
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])

for word_id, pos_tag, position in zip(
    range(len(words)),
    pos_tags,
    first_subword_positions
):
    print(f"{words[word_id]:12} -> {pos_tag:8} -> {tokens[position]}")

In [ ]:
# Extracting representations from one sentence

example = train_dataset[0]
words = example["tokens"]

encoded = tokenizer(
    words,
    is_split_into_words = True,
    return_tensors = "pt",
)

#move the input tensors to the same device as BERT
encoded = {key: value.to(device) for key, value in encoded.items()}

#Extract representations without calculating gradients
with torch.no_grad():
    outputs = model(**encoded, output_hidden_states=True)

print("Input token shape:", encoded["input_ids"].shape)
print("Final hidden-state shape:", outputs.last_hidden_state.shape)

In [ ]:
print("Number of hidden-state sets:", len(outputs.hidden_states))
print("Embedding shape:", outputs.hidden_states[0].shape)
print("Final layer shape:", outputs.hidden_states[-1].shape)

In [ ]:
#Get word IDs from the tokenizer for alignment
alignment = tokenizer(
    words,
    is_split_into_words = True,
    return_tensors = None,
)

word_ids = alignment.word_ids()

print("Number of original words:", len(words))
print("Number of BERT tokens:", len(word_ids))
print("Word IDs:", word_ids)

In [ ]:
# Find the first subword  position for each original word

first_subword_positions = []
previous_word_id = None

for position, word_id in enumerate(word_ids):
    if word_id is not None and word_id != previous_word_id:
        first_subword_positions.append(position)
    previous_word_id = word_id


# Extract word representations from all 12 Transformer layers
word_representations_by_layer = {}

for layer in range(1, 13):
    hidden_states = outputs.hidden_states[layer][0]
    word_representations_by_layer[layer] = hidden_states[first_subword_positions]

print("Number of layers:", len(word_representations_by_layer))
print("Layer 1 shape:", word_representations_by_layer[1].shape)
print("Layer 12 shape:", word_representations_by_layer[12].shape)



## Preparing dependency-relation labels

In [ ]:
example = train_dataset[0]

for i, (word, head, relation) in enumerate(
    zip(example["tokens"], example["head"], example["deprel"])
):
    print(
        f"Word index: {i},"
        f"Word: {word},"
        f" Head index: {head},"
        f"Relations: {relation}"
    )

In [ ]:
# verifying head of perticular word -(preacher)
example = train_dataset[0]

dependent_index = example["tokens"].index("preacher")
head_index = example["head"][dependent_index]

print("Dependent:", example["tokens"][dependent_index])
print("Head index in dataset:", head_index)
print("Head word:", example["tokens"][head_index - 1])
print("Relation:", example["deprel"][dependent_index])
print("Index:",example["tokens"].index("preacher"))

In [ ]:
#Select the representations from BERT's final Transformer layer
word_representations = word_representations_by_layer[12]
print("Word representations shape:", word_representations.shape)
preacher_index = example["tokens"].index("preacher")
preacher_representation = word_representations[preacher_index]
print("Representation shape:", preacher_representation.shape)

In [ ]:
# Enable all hidden states

model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_hidden_states = True,
)


In [ ]:
print("Model exists:", model is not None)
print("Hidden states enabled:", model.config.output_hidden_states)
print("Outputs exist:", outputs is not None)
print("Hidden states:", outputs.hidden_states if outputs is not None else "No outputs")

## Test tokenization and word alignment

In [ ]:
batch_size = 16

batch_examples = [train_dataset[i] for i in range(batch_size)]
batch_words = [example["tokens"] for example in batch_examples]

batch_encoding = tokenizer(
    batch_words,
    is_split_into_words=True,
    padding=True,
    return_tensors="pt",
).to(device)

print("Number of sentences:", len(batch_words))
print("Input shape:", batch_encoding["input_ids"].shape)
print("Attention mask shape:", batch_encoding["attention_mask"].shape)

for i, words in enumerate(batch_words):
    word_ids = batch_encoding.word_ids(batch_index=i)
    aligned_word_ids = {
        word_id for word_id in word_ids if word_id is not None
    }

    print(
        f"Sentence {i + 1}: "
        f"{len(words)} original words, "
        f"{len(aligned_word_ids)} aligned words"
    )

In [ ]:
#Check model and input devices
print("Model device:", next(model.parameters()).device)

for key, value in batch_encoding.items():
    print(f"{key} device:", value.device)

In [ ]:
#Move the existing model to the selected device
model = model.to(device)
model.eval()

print("Model device:", next(model.parameters()).device)
print("Input device:", batch_encoding["input_ids"].device)

## Extracting representations for the full training dataset

In [ ]:

batch_size = 16
batch_representations = {}

for start_idx in range(0, len(train_dataset), batch_size):
    end_idx = min(start_idx + batch_size, len(train_dataset))

    #Select the current batch
    batch_examples = [
        train_dataset[i]
        for i in range(start_idx, end_idx)
    ]

    #Extract words
    batch_words = [example["tokens"] for example in batch_examples]

    #Tokenize and move inputs to the device
    batch_encoding = tokenizer(
        batch_words,
        is_split_into_words = True,
        padding = True,
        return_tensors="pt",
    ).to(device)

    #Run BERT
    with torch.no_grad():
        outputs = model(
            **batch_encoding,
            output_hidden_states = True,
        )

    #Extract first-subword representations
    for i, words in enumerate(batch_words):
        global_idx = start_idx + i
        word_ids = batch_encoding.word_ids(batch_index=i)

        first_subword_positions = []
        previous_word_id = None

        for position, word_id in enumerate(word_ids):
            if word_id is not None and word_id != previous_word_id:
                first_subword_positions.append(position)
            previous_word_id = word_id

        positions = torch.tensor(
            first_subword_positions,
            dtype = torch.long,
            device = outputs.hidden_states[1].device,
        )

        batch_representations[global_idx] = {}

        for layer in range(1, 13):
            hidden_states = outputs.hidden_states[layer][i]
            batch_representations[global_idx][layer] = (
                hidden_states.index_select(0, positions).cpu()
            )

    #Release batch-specific GPU outputs before processing the next batch

    del outputs, batch_encoding

    batch_number = start_idx // batch_size + 1

    #if(batch_number + 1) % 10 == 0 or end_idx == len(train_dataset):
    print(
        f"Batch {batch_number}: "
        f" sentences {start_idx} - {end_idx - 1} | "
        f"Processed {end_idx} / {len(train_dataset)} sentences"
    )

print("sentences stored:", len(batch_representations))



In [ ]:
print("Sentences stored:", len(batch_representations))
print("Layers for sentence 0:", len(batch_representations[0]))
print("Sentence 0, layer 1 shape:", batch_representations[0][1].shape)
print("Representation device:", batch_representations[0][1].device)

In [ ]:
# creating a function to extract representations from validation dataset and the test dataset

def extract_representations(dataset, batch_size=16):
    batch_representations = {}

    for start_idx in range(0, len(dataset), batch_size):
        end_idx= min(start_idx + batch_size, len(dataset))


        #Selecting the sentences and extractig their words
        batch_examples = [
            dataset[i]
            for i in range(start_idx, end_idx)
        ]

        batch_words = [
            example["tokens"]
            for example in batch_examples
        ]

        #Tokenize the batch
        batch_encoding = tokenizer(
            batch_words,
            is_split_into_words = True,
            padding = True,
            return_tensors="pt",

        ).to(device)

        #Get hidden states from BERT
        with torch.no_grad():
            outputs = model(
                **batch_encoding,
                output_hidden_states = True,
            )

        #Selecting the first subword of each word
        for i, words in enumerate(batch_words):
            global_idx = start_idx + i
            word_ids = batch_encoding.word_ids(batch_index=i)

            first_subword_positions = []
            previous_word_id = None

            for position, word_id in enumerate(word_ids):
                if word_id is not None and word_id != previous_word_id:
                    first_subword_positions.append(position)
                previous_word_id = word_id

            #Converting positions into a PyTorch tensor
            positions = torch.tensor(
                first_subword_positions,
                dtype = torch.long,
                device = outputs.hidden_states[1].device,
                )

            #creating a dictionary for each setnece
            batch_representations[global_idx] = {}

            #Extract and store the vectors from all 12 layers
            for layer in range(1, 13):
                hidden_states = outputs.hidden_states[layer][i]         #Select hidden states for one sent
                batch_representations[global_idx][layer] = (
                hidden_states.index_select(0, positions).cpu()          #select one word per original word and store the vectors on CPU
                )

        del outputs, batch_encoding
    return batch_representations




# Extracting validation representations

In [ ]:
validation_representations = extract_representations(
    validation_dataset,
    batch_size=16
)

print("Validation sentences:", len(validation_representations))
print("Layers per sentence:", len(validation_representations[0]))

# Extracting test representations

In [ ]:
test_representations = extract_representations(
    test_dataset,
    batch_size=16
)

print("Test sentences:", len(test_representations))
print("Layers per sentence:", len(test_representations[0]))

## PREPARING tHE PROBE DATA

In [ ]:
example = train_dataset[0]

print("Tokens:", example["tokens"])
print("Heads:", example["head"])
print("Dependency relations:", example["deprel"])



In [ ]:
example = train_dataset[0]

word_index = 5

print("Word:", example["tokens"][word_index])
print("Head index:", example["head"][word_index])
print("Dependency relation:", example["deprel"][word_index])

dependent_vector = batch_representations[0][1][word_index]
print("Dependent vector shape:", dependent_vector.shape)

In [ ]:
# Prepare the data for one word
example = train_dataset[0]

non_root_count = 0

for word_index, head_index in enumerate(example["head"]):
    if head_index != 0:
        non_root_count += 1

print("Total words:", len(example["tokens"]))
print("Non-root words:", non_root_count)
print("Root words:", example["head"].count(0))

#Collect dependent vectors and labels

X_sentence = []
y_sentence = []

for word_index, head_index in enumerate(example["head"]):
    if head_index == 0:
        continue

    dependent_vector = batch_representations[0][1][word_index]
    relation = example["deprel"][word_index]

    X_sentence.append(dependent_vector)
    y_sentence.append(relation)

print("Number of examples:", len(X_sentence))
print("Number of labels:", len(y_sentence))

#Check the vector and label
print("First vector shape:", X_sentence[0].shape)
print("First label:", y_sentence[0])

#Preparing data for 12 layers
X_by_layer = {}
y_by_layer = {}
for layer in range(1, 13):
    X_by_layer[layer] = []
    y_by_layer[layer] = [] 
for sentence_idx in range(len(train_dataset)):
    example = train_dataset[sentence_idx]

    for word_index, head_index in enumerate(example["head"]):
        if head_index == 0:
            continue
        for layer in range(1, 13):
            dependent_vector = batch_representations[sentence_idx][layer][word_index]
            relation = example["deprel"][word_index]

            X_by_layer[layer].append(dependent_vector)
            y_by_layer[layer].append(relation)


print("Layer 1 vectors:", len(X_by_layer[1]))
print("Layer 1 labels:", len(y_by_layer[1]))
print("Layer 12 vectors:", len(X_by_layer[12]))
print("Layer 12 labels:", len(y_by_layer[12]))

In [ ]:
from collections import Counter

label_counts = Counter(y_by_layer[1])

print("Number of unique dependency relations:", len(label_counts))
print(label_counts.most_common(10))

In [ ]:
#Check the prepared training data
print("Number of BERT layers:", len(X_by_layer))
for layer in [1, 6, 12]:
    print(
        f"Layer {layer}:"
        f"{len(X_by_layer[layer])} vectors,"
        f"{len(y_by_layer[layer])} labels,"
        f"vector shape = {X_by_layer[layer][0].shape}"
    )

print("\n First 10 dependency labels:")
print(y_by_layer[1][:10])

print("\n Number of unique dependency relations:")
print(len(set(y_by_layer[1])))

In [ ]:
#Prepare the data for Logistic Regression

import numpy as np
from sklearn.preprocessing import LabelEncoder

#Select BERT layer 1 for our first probe
X_train = torch.stack(X_by_layer[1]).numpy()

#Convert dependency labels into numerical IDs
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_by_layer[1])

print("Feature matrix shape:", X_train.shape)
print("Label array shape:", y_train.shape)
print("Number of classes:", len(label_encoder.classes_))
print("First 10 encoded labels:", y_train[:10])

## Split off Validation data and train Logistic Regression


In [ ]:
#Prepare validation features and labels for layer 1

X_val_list = []
y_val_list = []

for sentence_idx, example in enumerate(validation_dataset):
    for word_index, head_index in enumerate(example["head"]):
        #Exclude root words, as training data
        if head_index == 0:
            continue

        vector = validation_representations[sentence_idx][1][word_index]
        relation = example["deprel"][word_index]

        X_val_list.append(vector)
        y_val_list.append(relation)

X_val = torch.stack(X_val_list).numpy()

#use the mapping learned from the training labels
y_val = label_encoder.transform(y_val_list)

print("Validation feature matrix:", X_val.shape)
print("Validation label array:", y_val.shape)

all_labels = np.arange(len(label_encoder.classes_))

## Train the Classifier

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

#Create the probe classiifer
probe_layer1 = LogisticRegression(
    max_iter= 1000,
    solver = "lbfgs"
)

#Train the probe using Layer 1 representations
probe_layer1.fit(X_train, y_train)

print("Layer 1 probe training completed!")

In [ ]:
## Evaluate the Layer 1 probe

# Predict dependency relations for validation examples
y_val_pred = probe_layer1.predict(X_val)

# Calculate evaluation metrics
val_accuracy = accuracy_score(y_val, y_val_pred)

val_macro_f1 = f1_score(
    y_val,
    y_val_pred,
    labels=all_labels,
    average="macro",
    zero_division=0
)

print(f"Layer 1 Validation Accuracy: {val_accuracy:.4f}")
print(f"Layer 1 Validation Macro-F1: {val_macro_f1:.4f}")

In [ ]:
#Examining which dependency relations are difficult
from sklearn.metrics import classification_report

print(
    classification_report(
        y_val,
        y_val_pred,
        labels = range(len(label_encoder.classes_)),
        target_names = label_encoder.classes_,
        zero_division=0
    )
)

In [ ]:
# Verifying macro-F1 discrepancy

print("Macro=F1:", f1_score(
    y_val, y_val_pred,
    average="macro",
    zero_division = 0
))
print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("Number of validation predictions:", len(y_val_pred))
print("Number of validation labels:", len(y_val))

In [ ]:
# Most- frequent class baseline

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

#Always predict the most frequent training class
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

#Predict labels for the same validation examples as the probe
baseline_predictions = baseline.predict(X_val)

#Evaluate using the same fixed set of 50 classes
all_labels = np.arange(len(label_encoder.classes_))

baseline_accuracy = accuracy_score(y_val, baseline_predictions)
baseline_macro_f1 = f1_score(
    y_val,
    baseline_predictions,
    labels=all_labels,
    average="macro",
    zero_division=0
)

print(f"Baseline class: {label_encoder.inverse_transform([baseline.classes_[0]])[0]}")
print(f"Baseline validation accuracy: {baseline_accuracy:.4f}")
print(f"Baseline validation macro-F1: {baseline_macro_f1:.4f}")

## Train and evaluate on the remaining layers


In [ ]:
#Store the Layer 1 results we calculated
results = [{
    "layer": 1,
    "accuracy": val_accuracy,
    "macro_f1": val_macro_f1
}]

#Train and evaluate Layers 2-12

for layer in range(2, 13):
    print(f"Trianing probe for Layer {layer}...")

    #Prepare training features
    X_train_layer = torch.stack(X_by_layer[layer]).numpy()

    #Prepare validation features
    X_val_list_layer = []

    for sentence_idx, example in enumerate(validation_dataset):
        for word_index, head_index in enumerate(example["head"]):
            if head_index == 0:
                continue

            vector = validation_representations[sentence_idx][layer][word_index]
            X_val_list_layer.append(vector)


    X_val_layer = torch.stack(X_val_list_layer).numpy()

    #train the probe
    probe = LogisticRegression(max_iter=1000, solver="lbfgs")
    probe.fit(X_train_layer, y_train)

    #Evaluate on validation data
    predictions = probe.predict(X_val_layer)

    accuracy = accuracy_score(y_val, predictions)
    macro_f1 = f1_score(
        y_val, predictions,
        average = "macro",
        zero_division=0
    )

    results.append({
        "layer": layer,
        "accuracy": accuracy,
        "macro_f1": macro_f1
    })

    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Macro_f1: {macro_f1:.4f}")

#Display results for all 12 layers
results_df = pd.DataFrame(results)
display(results_df)

## Combine Head and Depenedent representations

In [ ]:
# Prepare the training data for Layer 1

X_head_dep_layer1 = []
y_head_dep_layer1= []

for sentence_idx, example in enumerate(train_dataset):
    for word_index, head_index in enumerate(example["head"]):

        #skip root words
        if head_index == 0:
            continue

        #Get the head and dependent vectors from Layer 1
        head_vector = batch_representations[sentence_idx][1][head_index - 1]
        dependent_vector = batch_representations[sentence_idx][1][word_index]

        #Join the two vectors
        combined_vector = torch.cat([head_vector, dependent_vector])

        X_head_dep_layer1.append(combined_vector)
        y_head_dep_layer1.append(example["deprel"][word_index])


X_head_dep_layer1 = torch.stack(X_head_dep_layer1).numpy()
print("Feature shape:", X_head_dep_layer1.shape)
print("Number of labels:", len(y_head_dep_layer1))

In [ ]:
#Prepare validation data
X_head_dep_val_list = []
y_head_dep_val_list = []

for sentence_idx, example in enumerate(validation_dataset):
    for word_index, head_index in enumerate(example["head"]):

        #Skip root words
        if head_index == 0:
            continue

        #Get head and dependent vectors from Layer 1
        head_vector = validation_representations[sentence_idx][1][head_index - 1]
        dependent_vector = validation_representations[sentence_idx][1][word_index]

        #Combine both vectors
        combined_vector = torch.cat([head_vector, dependent_vector])

        X_head_dep_val_list.append(combined_vector)
        y_head_dep_val_list.append(example["deprel"][word_index])


#Convert feature vectors into a Numpy array
X_head_dep_val = torch.stack(X_head_dep_val_list).numpy()

y_head_dep_val = label_encoder.transform(y_head_dep_val_list)

print("Validation feature shape:", X_head_dep_val.shape)
print("Validation label shape:", y_head_dep_val.shape)


In [ ]:
#Train and evaluate the probe

# Prepare training features
X_head_dep_train = X_head_dep_layer1

# Encode training labels with the existing encoder
y_head_dep_train = label_encoder.transform(y_head_dep_layer1)

# Train the probe
probe_head_dep_layer1 = LogisticRegression(
    max_iter=1000,
    solver="lbfgs"
)

probe_head_dep_layer1.fit(X_head_dep_train, y_head_dep_train)

# Predict validation relations
y_head_dep_pred = probe_head_dep_layer1.predict(X_head_dep_val)

# Evaluate
all_labels = np.arange(len(label_encoder.classes_))

head_dep_accuracy = accuracy_score(
    y_head_dep_val,
    y_head_dep_pred
)

head_dep_macro_f1 = f1_score(
    y_head_dep_val,
    y_head_dep_pred,
    labels=all_labels,
    average="macro",
    zero_division=0
)

print(f"Layer 1 Head + Dependent Accuracy: {head_dep_accuracy:.4f}")
print(f"Layer 1 Head + Dependent Macro-F1: {head_dep_macro_f1:.4f}")

In [ ]:
#checking Feature data

print("Training features:", X_head_dep_train.shape)
print("Training labels:", y_head_dep_train.shape)

print("Validation features:", X_head_dep_val.shape)
print("Validation labels:", y_head_dep_val.shape)

print("Training feature type:", type(X_head_dep_train))
print("Validation feature type:", type(X_head_dep_val))

print("Training labels match feature rows:",
      len(y_head_dep_train) == X_head_dep_train.shape[0])

print("Validation labels match feature rows:",
      len(y_head_dep_val) == X_head_dep_val.shape[0])

print("Training features contain NaN:",
      np.isnan(X_head_dep_train).any())

print("Validation features contain NaN:",
      np.isnan(X_head_dep_val).any())

In [ ]:
#Prepare the head + dependent experiment for all layers

X_head_dep_train_by_layer = {}
y_head_dep_train_by_layer = {}

X_head_dep_val_by_layer = {}
y_head_dep_val_by_layer = {}

for layer in range(1, 13):
    # Training features
    train_features = []
    train_labels = []

    for sentence_idx, example in enumerate(train_dataset):
        for word_index, head_index in enumerate(example["head"]):
            # Skip root words: they do not have an ordinary head
            if head_index == 0:
                continue

            head_vector = batch_representations[sentence_idx][layer][head_index - 1]
            dependent_vector = batch_representations[sentence_idx][layer][word_index]

            combined_vector = torch.cat([head_vector, dependent_vector])

            train_features.append(combined_vector)
            train_labels.append(example["deprel"][word_index])

    X_head_dep_train_by_layer[layer] = torch.stack(train_features).numpy()
    y_head_dep_train_by_layer[layer] = label_encoder.transform(train_labels)

    # Validation features
    val_features = []
    val_labels = []

    for sentence_idx, example in enumerate(validation_dataset):
        for word_index, head_index in enumerate(example["head"]):
            if head_index == 0:
                continue

            head_vector = validation_representations[sentence_idx][layer][head_index - 1]
            dependent_vector = validation_representations[sentence_idx][layer][word_index]

            combined_vector = torch.cat([head_vector, dependent_vector])

            val_features.append(combined_vector)
            val_labels.append(example["deprel"][word_index])

    X_head_dep_val_by_layer[layer] = torch.stack(val_features).numpy()
    y_head_dep_val_by_layer[layer] = label_encoder.transform(val_labels)

    print(
        f"Layer {layer}: "
        f"train={X_head_dep_train_by_layer[layer].shape}, "
        f"validation={X_head_dep_val_by_layer[layer].shape}"
    )

In [ ]:
# Check the Layer 8 head + dependent data

print("Training features:", X_head_dep_train_by_layer[8].shape)
print("Training labels:", y_head_dep_train_by_layer[8].shape)

print("Validation features:", X_head_dep_val_by_layer[8].shape)
print("Validation labels:", y_head_dep_val_by_layer[8].shape)

In [ ]:
head_dep_results = []
all_labels = np.arange(len(label_encoder.classes_))

for layer in range(1, 13):
    print(f"Training head + dependent probe for Layer {layer}...")

    X_train = X_head_dep_train_by_layer[layer]
    y_train = y_head_dep_train_by_layer[layer]
    X_val = X_head_dep_val_by_layer[layer]
    y_val = y_head_dep_val_by_layer[layer]

    probe = LogisticRegression(max_iter=1000, solver="lbfgs")
    probe.fit(X_train, y_train)

    y_pred = probe.predict(X_val)

    head_dep_results.append({
        "layer": layer,
        "accuracy": accuracy_score(y_val, y_pred),
        "macro_f1": f1_score(
            y_val,
            y_pred,
            labels=all_labels,
            average="macro",
            zero_division=0
        )
    })

head_dep_results_df = pd.DataFrame(head_dep_results)
display(head_dep_results_df.round(4))

In [ ]:
# Train and retain the selected Layer 8 probe

probe_head_dep_layer8 = LogisticRegression(
    max_iter=1000,
    solver="lbfgs"
)

probe_head_dep_layer8.fit(
    X_head_dep_train_by_layer[8],
    y_head_dep_train_by_layer[8]
)

print("Layer 8 probe is ready for test evaluation.")

In [ ]:
# Prepare test features for Layer 8 (head + dependent)

X_test_head_dep_layer8 = []
y_test_head_dep_layer8 = []

for sentence_idx, example in enumerate(test_dataset):
    for word_index, head_index in enumerate(example["head"]):

        # Skip root words
        if head_index == 0:
            continue

        # Get head and dependent vectors from Layer 8
        head_vector = test_representations[sentence_idx][8][head_index - 1]
        dependent_vector = test_representations[sentence_idx][8][word_index]

        # Combine head and dependent vectors
        combined_vector = torch.cat([head_vector, dependent_vector])

        X_test_head_dep_layer8.append(combined_vector)
        y_test_head_dep_layer8.append(example["deprel"][word_index])

# Convert features to NumPy
X_test_head_dep_layer8 = torch.stack(X_test_head_dep_layer8).numpy()

# Encode labels using the encoder fitted on training labels
y_test_head_dep_layer8 = label_encoder.transform(y_test_head_dep_layer8)

print("Test feature shape:", X_test_head_dep_layer8.shape)
print("Test label shape:", y_test_head_dep_layer8.shape)

## Evaluate a trained probe and calculate test accuracy and macro-f1

In [ ]:
# Inspect one test example
print(test_dataset[0])

# Inspect its columns
print(test_dataset.column_names)

# Check the number of test sentences
print("Test sentences:", len(test_dataset))

In [ ]:
# Check dependency labels

# Count non-root dependency relations in the test set
test_labels = [
    relation
    for sentence in test_dataset
    for head, relation in zip(sentence["head"], sentence["deprel"])
    if head != 0
]

print("Number of non-root tokens:", len(test_labels))
print("Unique dependency relations:", len(set(test_labels)))
print("Test dependency labels:", sorted(set(test_labels)))

In [ ]:
# Compare test labels with training labels

# Use the LabelEncoder already fitted on training labels
train_label_set = set(label_encoder.classes_)
test_label_set = set(test_labels)

print("Number of training classes:", len(train_label_set))
print("Number of test classes:", len(test_label_set))

print(
    "Test labels not seen during training:",
    sorted(test_label_set - train_label_set)
)

print(
    "Training labels absent from test:",
    sorted(train_label_set - test_label_set)
)

In [ ]:
"""# Show variables related to probes, features, and labels
for name, value in globals().copy().items():
    if any(term in name.lower() for term in [
        "probe", "classifier", "model",
        "x_train", "x_val", "x_test",
        "y_train", "y_val", "y_test"
    ]):
        if not name.startswith("_"):
            print(name, type(value).__name__)"""

In [ ]:
# Inspect the selected Layer 8 probe and test features
print("Probe input features:", probe_head_dep_layer8.n_features_in_)
print("Test feature shape:", X_test_head_dep_layer8.shape)
print("Number of test labels:", len(y_test_head_dep_layer8))
print("Number of feature rows:", X_test_head_dep_layer8.shape[0])

print(
    "Feature dimensions match:",
    probe_head_dep_layer8.n_features_in_ == X_test_head_dep_layer8.shape[1]
)

print(
    "Feature rows match labels:",
    X_test_head_dep_layer8.shape[0] == len(y_test_head_dep_layer8)
)


In [ ]:
# Final test evaluation: Layer 8 head + dependent probe

y_test_pred = probe_head_dep_layer8.predict(
    X_test_head_dep_layer8
)

test_accuracy = accuracy_score(
    y_test_head_dep_layer8,
    y_test_pred
)

test_macro_f1 = f1_score(
    y_test_head_dep_layer8,
    y_test_pred,
    labels=np.arange(len(label_encoder.classes_)),
    average="macro",
    zero_division=0
)

print(f"Layer 8 Test Accuracy: {test_accuracy:.4f}")
print(f"Layer 8 Test Macro-F1: {test_macro_f1:.4f}")

In [ ]:
# Generate a report for each dependency relation
report = classification_report(
    y_test_head_dep_layer8,
    y_test_pred,
    labels=np.arange(len(label_encoder.classes_)),
    target_names=label_encoder.classes_,
    zero_division=0,
    output_dict=True
)

# Convert the report into a readable table
report_df = pd.DataFrame(report).T

# Display per-class results, excluding summary rows
display(report_df.drop(
    index=["accuracy", "macro avg", "weighted avg"],
    errors="ignore"
).round(4))

In [ ]:
# Compare macro-F1 evaluation conventions

all_training_labels = np.arange(len(label_encoder.classes_))

observed_test_labels = np.union1d(
    y_test_head_dep_layer8,
    y_test_pred
)

macro_f1_all_labels = f1_score(
    y_test_head_dep_layer8,
    y_test_pred,
    labels=all_training_labels,
    average="macro",
    zero_division=0
)

macro_f1_observed_labels = f1_score(
    y_test_head_dep_layer8,
    y_test_pred,
    labels=observed_test_labels,
    average="macro",
    zero_division=0
)

print(f"Macro-F1 over all training labels: {macro_f1_all_labels:.4f}")
print(f"Macro-F1 over observed test/predicted labels: {macro_f1_observed_labels:.4f}")
print("Training labels:", len(all_training_labels))
print("Observed test/predicted labels:", len(observed_test_labels))

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Find the 10 most frequent gold dependency relations in the test set
top_10_labels = (
    pd.Series(y_test_head_dep_layer8)
    .value_counts()
    .head(10)
    .index
    .to_numpy()
)

# Calculate the confusion matrix for these relations
cm = confusion_matrix(
    y_test_head_dep_layer8,
    y_test_pred,
    labels=top_10_labels
)

# Get readable dependency relation names
top_10_names = label_encoder.inverse_transform(top_10_labels)

# Plot the confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=top_10_names,
    yticklabels=top_10_names
)

plt.xlabel("Predicted dependency relation")
plt.ylabel("True dependency relation")
plt.title("Layer 8 Head + Dependent Probe: Top 10 Relations")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Row-normalized confusion matrix

cm_normalized = confusion_matrix(
    y_test_head_dep_layer8,
    y_test_pred,
    labels=top_10_labels,
    normalize="true"
)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".1%",
    cmap="Blues",
    xticklabels=top_10_names,
    yticklabels=top_10_names
)

plt.xlabel("Predicted dependency relation")
plt.ylabel("True dependency relation")
plt.title("Layer 8 Probe: Normalized Confusion Matrix")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Find examples where true amod is predicted as compound

true_labels = label_encoder.inverse_transform(
    y_test_head_dep_layer8
)

predicted_labels = label_encoder.inverse_transform(
    y_test_pred
)

count = 0

for sentence_idx, example in enumerate(test_dataset):
    for word_index, head_index in enumerate(example["head"]):
        if head_index == 0:
            continue

        # Locate this token in the flattened test-label arrays
        # (we'll use a running index in the corrected version below)

In [ ]:
# Find examples where true amod is predicted as compound

true_labels = label_encoder.inverse_transform(y_test_head_dep_layer8)
predicted_labels = label_encoder.inverse_transform(y_test_pred)

token_idx = 0
count = 0

for sentence_idx, example in enumerate(test_dataset):
    words = example["tokens"]
    heads = example["head"]
    relations = example["deprel"]

    for word_index, head_index in enumerate(heads):
        if head_index == 0:
            continue

        true_relation = true_labels[token_idx]
        predicted_relation = predicted_labels[token_idx]
        token_idx += 1

        if true_relation == "amod" and predicted_relation == "compound":
            head_word = words[head_index - 1]

            print("Sentence:", " ".join(words))
            print("Head:", head_word)
            print("Dependent:", words[word_index])
            print("True relation:", true_relation)
            print("Predicted relation:", predicted_relation)
            print("-" * 60)

            count += 1
            if count == 5:
                break

    if count == 5:
        break

print("Examples found:", count)

In [ ]:
# Find examples where true compound is predicted as amod

token_idx = 0
count = 0

for sentence_idx, example in enumerate(test_dataset):
    words = example["tokens"]
    heads = example["head"]
    relations = example["deprel"]

    for word_index, head_index in enumerate(heads):
        if head_index == 0:
            continue

        true_relation = true_labels[token_idx]
        predicted_relation = predicted_labels[token_idx]
        token_idx += 1

        if true_relation == "compound" and predicted_relation == "amod":
            head_word = words[head_index - 1]

            print("Sentence:", " ".join(words))
            print("Head:", head_word)
            print("Dependent:", words[word_index])
            print("True relation:", true_relation)
            print("Predicted relation:", predicted_relation)
            print("-" * 60)

            count += 1
            if count == 5:
                break

    if count == 5:
        break

print("Examples found:", count)

In [ ]:
# Count confusion between amod and compound

from sklearn.metrics import confusion_matrix

amod_id = label_encoder.transform(["amod"])[0]
compound_id = label_encoder.transform(["compound"])[0]

cm_two = confusion_matrix(
    y_test_head_dep_layer8,
    y_test_pred,
    labels=[amod_id, compound_id]
)

print("Rows: true [amod, compound]")
print("Columns: predicted [amod, compound]")
print(cm_two)

## Compare misclassified and correctly classified examples

In [ ]:

true_labels = label_encoder.inverse_transform(y_test_head_dep_layer8)
predicted_labels = label_encoder.inverse_transform(y_test_pred)

token_idx = 0
wrong_examples = []
correct_examples = []

for sentence_idx, example in enumerate(test_dataset):
    words = example["tokens"]
    heads = example["head"]
    relations = example["deprel"]

    for word_index, head_index in enumerate(heads):
        if head_index == 0:
            continue

        true_relation = true_labels[token_idx]
        predicted_relation = predicted_labels[token_idx]
        token_idx += 1

        if true_relation == "amod":
            item = {
                "sentence": " ".join(words),
                "head": words[head_index - 1],
                "dependent": words[word_index],
                "true": true_relation,
                "predicted": predicted_relation,
            }

            if predicted_relation == "compound" and len(wrong_examples) < 5:
                wrong_examples.append(item)
            elif predicted_relation == "amod" and len(correct_examples) < 5:
                correct_examples.append(item)

print("MISCLASSIFIED amod EXAMPLES")
for item in wrong_examples:
    print(item, "\n")

print("CORRECTLY CLASSIFIED amod EXAMPLES")
for item in correct_examples:
    print(item, "\n")


In [ ]:

true_labels = label_encoder.inverse_transform(y_test_head_dep_layer8)
predicted_labels = label_encoder.inverse_transform(y_test_pred)

token_idx = 0
wrong_examples = []
correct_examples = []

for sentence_idx, example in enumerate(test_dataset):
    words = example["tokens"]
    heads = example["head"]
    relations = example["deprel"]

    for word_index, head_index in enumerate(heads):
        if head_index == 0:
            continue

        true_relation = true_labels[token_idx]
        predicted_relation = predicted_labels[token_idx]
        token_idx += 1

        if true_relation == "compound":
            item = {
                "sentence": " ".join(words),
                "head": words[head_index - 1],
                "dependent": words[word_index],
                "true": true_relation,
                "predicted": predicted_relation,
            }

            if predicted_relation == "amod" and len(wrong_examples) < 5:
                wrong_examples.append(item)
            elif predicted_relation == "compound" and len(correct_examples) < 5:
                correct_examples.append(item)

print("MISCLASSIFIED compound EXAMPLES")
for item in wrong_examples:
    print(item, "\n")

print("CORRECTLY CLASSIFIED compound EXAMPLES")
for item in correct_examples:
    print(item, "\n")


In [ ]:

amod_to_compound = 72
compound_to_amod = 65

true_amod_total = 1213
true_compound_total = 889

print(
    "amod predicted as compound:",
    f"{amod_to_compound / true_amod_total:.2%}"
)

print(
    "compound predicted as amod:",
    f"{compound_to_amod / true_compound_total:.2%}"
)


In [ ]:



dummy = DummyClassifier(strategy="most_frequent")

dummy.fit(
    X_head_dep_train_by_layer[8],
    y_head_dep_train_by_layer[8]
)

y_test_dummy_pred = dummy.predict(X_test_head_dep_layer8)

dummy_test_accuracy = accuracy_score(
    y_test_head_dep_layer8,
    y_test_dummy_pred
)

dummy_test_macro_f1 = f1_score(
    y_test_head_dep_layer8,
    y_test_dummy_pred,
    labels=np.arange(len(label_encoder.classes_)),
    average="macro",
    zero_division=0
)

print(f"Baseline test accuracy: {dummy_test_accuracy:.4f}")
print(f"Baseline test macro-F1: {dummy_test_macro_f1:.4f}")


In [ ]:

comparison_df = results_df.merge(
    head_dep_results_df,
    on="layer",
    suffixes=("_dependent", "_head_dep")
)

comparison_df["accuracy_improvement"] = (
    comparison_df["accuracy_head_dep"]
    - comparison_df["accuracy_dependent"]
)

comparison_df["macro_f1_improvement"] = (
    comparison_df["macro_f1_head_dep"]
    - comparison_df["macro_f1_dependent"]
)

display(comparison_df.round(4))


In [ ]:



# Plot 1: Validation accuracy
plt.figure(figsize=(10, 5))

plt.plot(
    comparison_df["layer"],
    comparison_df["accuracy_dependent"],
    marker="o",
    label="Dependent-only"
)

plt.plot(
    comparison_df["layer"],
    comparison_df["accuracy_head_dep"],
    marker="o",
    label="Head + dependent"
)

plt.xlabel("Transformer layer")
plt.ylabel("Validation accuracy")
plt.title("Dependency Relation Probing: Validation Accuracy")
plt.xticks(range(1, 13))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# Plot 2: Validation macro-F1
plt.figure(figsize=(10, 5))

plt.plot(
    comparison_df["layer"],
    comparison_df["macro_f1_dependent"],
    marker="o",
    label="Dependent-only"
)

plt.plot(
    comparison_df["layer"],
    comparison_df["macro_f1_head_dep"],
    marker="o",
    label="Head + dependent"
)

plt.xlabel("Transformer layer")
plt.ylabel("Validation macro-F1")
plt.title("Dependency Relation Probing: Validation Macro-F1")
plt.xticks(range(1, 13))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
